# Data Validation

Validação de qualidade da base crua disponibilizada pela locaweb
- Unicidade de registros
- Tipo e formato de dados
- Adequação às regras de negócio
- Tratamento de Nulos

## Configurações Iniciais

### Configurações a depender do ambiente

In [72]:
import os
import sys
import subprocess

# --- CONFIGURAÇÃO ALVO ---
TARGET_PYSPARK = "4.1.1"

# 1. Identifica o Ambiente (Fora da função para ser global)
IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "☁️ Google Colab" if IN_COLAB else "💻 Ambiente Local (WSL/Jupyter)"

print(f"Detectado: {ENV_NAME}")
print(f"Versão do Python: {sys.version.split()[0]}")

# 2. Define os Caminhos Globais
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/"
    SAVE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/1_bronze_data/"
else:
    BASE_PATH = "../0_raw_data/"
    SAVE_PATH = "../1_bronze_data/"
def setup_pyspark():
    if IN_COLAB:
        try:
            import pyspark
            if pyspark.__version__ != TARGET_PYSPARK:
                print(f"(!) Atualizando PySpark de {pyspark.__version__} para {TARGET_PYSPARK}...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])
                print("🚨 Reinicie o Ambiente (Runtime > Restart Session) para aplicar a mudança!")
        except ImportError:
            print(f"(!) Instalando PySpark {TARGET_PYSPARK}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])

    # Verificação Final
    try:
        import pyspark
        if pyspark.__version__ == TARGET_PYSPARK:
            print(f"✅ PySpark {pyspark.__version__} pronto!")
        else:
            print(f"⚠️ Alerta: PySpark está na versão {pyspark.__version__}. Alvo era {TARGET_PYSPARK}.")
    except ImportError:
        print("❌ Erro: PySpark não encontrado.")

setup_pyspark()

Detectado: 💻 Ambiente Local (WSL/Jupyter)
Versão do Python: 3.12.3
✅ PySpark 4.1.1 pronto!


### Importação de Bibliotecas

In [73]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import * # Para definir Schemas (StructType, DoubleType, etc)
from pyspark.sql.window import Window
import pandas as pd

### Inicialização da Sessão Spark

In [74]:
spark = SparkSession.builder \
    .appName("DataValidation") \
    .master("local[*]") \
    .getOrCreate()

In [75]:
spark

### Importação de Dados

In [76]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(f"{BASE_PATH}/LW-DATASET-CSV.CSV")

In [77]:
df.show(20)

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|INC8654273| 3 - Média|   NULL|     NULL|        NULL|         Team14|             IC00001|2025-12-31 23:45:18|     NULL|2025-12-31 23:45:32|     14|                NULL|Problem:

In [78]:
df.printSchema()

root
 |-- Número: string (nullable = true)
 |-- Prioridade: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Subcategoria: string (nullable = true)
 |-- Grupo designado: string (nullable = true)
 |-- Item de configuração: string (nullable = true)
 |-- Aberto: timestamp (nullable = true)
 |-- Resolvido: timestamp (nullable = true)
 |-- Encerrado: timestamp (nullable = true)
 |-- Duração: integer (nullable = true)
 |-- Código de fechamento: string (nullable = true)
 |-- Descrição resumida: string (nullable = true)
 |-- Solução: string (nullable = true)
 |-- Aberto por: string (nullable = true)
 |-- Incidente Pai: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Entrou para KPI?: string (nullable = true)
 |-- KPI Violado?: string (nullable = true)



## Validação da Base

In [79]:
def identificar_nulos(df:DataFrame, nome_coluna: str) -> DataFrame:
    """
    Adiciona uma coluna booleana indicando se o valor na coluna informada é nulo.
    """
    return df.withColumn(
        f"{nome_coluna}_is_null", 
        F.col(nome_coluna).isNull()
    )

In [80]:
def identificar_valores_fora_do_padrao(df: DataFrame, nome_coluna: str, padrao: list) -> DataFrame:
    """
    Adiciona uma coluna booleana indicando se o valor na coluna informada está fora do padrão esperado.
    O parâmetro 'padrao' pode ser uma função ou expressão que define o critério de validação.
    """
    expressao_validacao = F.col(nome_coluna).isin(padrao)
    return df.withColumn(
        f"{nome_coluna}_is_out_of_pattern", 
        ~expressao_validacao
    )

In [81]:
# Criação de Coluna "Deve ser retirado da base" e "Modified_Record"
df = df\
    .withColumn(
        "Drop_from_base",
        F.lit(False)
    )\
    .withColumn(
        "Modified_Record",
        F.lit(False)
    )

In [82]:
colunas_originais = df.columns
total_registros = df.count()

In [83]:
filtro_registros = F.col("Drop_from_base") == False

### PK - Formato e Duplicatas

In [84]:
# Verificação de formato da coluna "Número" (PK)
"""
Valida se a coluna segue o padrão 'INC' seguido de exatamente 7 dígitos.
Exemplo: INC0001234
"""
# ^      : Início da string
# INC    : Prefixo literal
# \d{7}  : Exatamente 7 dígitos numéricos
# $      : Fim da string
padrao_regex = r"^INC\d{7}$"
df = df.withColumn(
    "Número_is_invalid", 
    ~F.col("Número").rlike(padrao_regex)
)
filtro = F.col("Número_is_invalid") == True
total_invalidos = df.filter(filtro).count()
pct_invalidos = (total_invalidos / total_registros)
if pct_invalidos > 0.005:
    print("⚠️ Atenção: Mais de 0.5% dos registros estão com 'Número' fora do padrão. Recomendado revisar a fonte dos dos dados.")
else:
    print("✅ A maioria dos registros segue o padrão esperado para 'Número'.")
    df = df.withColumn(
        "Drop_from_base", 
        F.when(F.col("Número_is_invalid"), True).otherwise(False)
    )
print(f"Total de registros com 'Número' fora do padrão: {total_invalidos} ({pct_invalidos*100:.2f}%)")
df.filter(filtro).show(total_invalidos, truncate=False)

# Drop da coluna de validação
df = df.drop("Número_is_invalid")


✅ A maioria dos registros segue o padrão esperado para 'Número'.
Total de registros com 'Número' fora do padrão: 11 (0.01%)
+-------------------------------------------------------------------------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------+-------+--------------------+------------------+-------+----------+-------------+------+----------------+------------+--------------+---------------+-----------------+
|Número                                                                   |Prioridade|Produto      |Categoria |Subcategoria   |Grupo designado|Item de configuração|Aberto|Resolvido|Encerrado|Duração|Código de fechamento|Descrição resumida|Solução|Aberto por|Incidente Pai|Status|Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|Número_is_invalid|
+-------------------------------------------------------------------------+----------+-------------+----------+---------------+---------------+---------------

In [85]:
total_registros = df.filter(filtro_registros).count()
total_registros

122543

In [86]:
# Verificação de Unicidade da PK - desconsiderando registros previamente marcados para remoção
chaves_distintas = df.filter(filtro_registros).select("Número").distinct().count()
unicidade_chave = total_registros == chaves_distintas
if unicidade_chave:
    print("✅ A coluna 'Número' é única.")
else:
    print("⚠️ A coluna 'Número' contém duplicatas.")
    print(f"Número de duplicatas: {total_registros - chaves_distintas} ({(total_registros - chaves_distintas)/total_registros*100:.2f}%)")
    # Adicionar uma coluna booleana indicando se o valor na coluna informada é duplicado.
    window_spec = Window.partitionBy("Número")
    df = df.withColumn(
        "Número_is_duplicate", 
        F.count("Número").over(window_spec) > 1
    )
    df.filter(F.col("Número_is_duplicate") == True).orderBy("Número").show(20)

✅ A coluna 'Número' é única.


### Valores Permitidos

In [87]:
# Validação de lista de valores permitidos por coluna
colunas_com_valores_definidos = {
    "Prioridade": ["1 - Crítica","2 - Alta","3 - Média","4 - Baixa","5 - Muito Baixa"],
    "Solução": ["Contorno", "Definitiva"],
    "Aberto por": ["Manual", "Monitoramento"],
    "Status": ["Aguardando Problema", "Encerrado", "Encerrado Automaticamente", "Sem Intervenção"],
    "Entrou para KPI?": ["SIM", "NAO"],
    "KPI Violado?": ["SIM", "NAO", "N/A"]
}

for coluna, valores_permitidos in colunas_com_valores_definidos.items():
    df = identificar_valores_fora_do_padrao(df, coluna, valores_permitidos)
    filtro_out_of_pattern = F.col(f"{coluna}_is_out_of_pattern") == True
    total_fora_do_padrao = df.filter(filtro_out_of_pattern).count()
    pct_fora_do_padrao = (total_fora_do_padrao / total_registros)
    if total_fora_do_padrao > 0:
        print(f"⚠️ Atenção: Total de registros com '{coluna}' fora do padrão: {total_fora_do_padrao} ({pct_fora_do_padrao*100:.2f}%)")
        df.filter(filtro_out_of_pattern).show(total_fora_do_padrao, truncate=False)
    else:
        print(f"✅ Todos os registros seguem o padrão esperado para '{coluna}'.")
    # Drop da coluna auxiliar
    df = df.drop(f"{coluna}_is_out_of_pattern")

✅ Todos os registros seguem o padrão esperado para 'Prioridade'.
✅ Todos os registros seguem o padrão esperado para 'Solução'.
✅ Todos os registros seguem o padrão esperado para 'Aberto por'.
✅ Todos os registros seguem o padrão esperado para 'Status'.
✅ Todos os registros seguem o padrão esperado para 'Entrou para KPI?'.
✅ Todos os registros seguem o padrão esperado para 'KPI Violado?'.


### Regras de Negócio

#### Categoria e Subcategoria

In [88]:
# Validando o se todas as subcategorias preenchidas tem a categoria preenchida também
df = df\
    .withColumn(
        "Subcategoria_is_filled", 
        F.col("Subcategoria").isNotNull()
    ).withColumn(
        "Categoria_is_filled", 
        F.col("Categoria").isNotNull()
    ).withColumn(
        "Subcategoria_without_Categoria", 
        F.when(
            (F.col("Subcategoria_is_filled") == True) & (F.col("Categoria_is_filled") == False), 
            True
        ).otherwise(False)
    )
total_subcategoria_sem_categoria = df.filter(filtro_registros).filter(F.col("Subcategoria_without_Categoria") == True).count()
if total_subcategoria_sem_categoria > 0:
    print(f"⚠️ Encontrados {total_subcategoria_sem_categoria} registros onde 'Subcategoria' está preenchida mas 'Categoria' está nula. Recomendado revisar esses casos.")
    linhas_problematicas = df.filter(filtro_registros).filter(F.col("Subcategoria_without_Categoria") == True)
    linhas_problematicas.show(20, truncate=False)

    # Preenchendo a coluna "Categoria" baseado na subcategoria e item de configuração
    filtro_sub_e_ic = linhas_problematicas.select("Subcategoria", "Item de configuração").distinct()
    base_reduzida = df.filter(
        (F.col("Categoria").isNotNull()) &
        (F.col("Subcategoria").isNotNull()) &
        (F.col("Item de configuração").isNotNull())
    ).join(
        filtro_sub_e_ic,
        on=["Subcategoria", "Item de configuração"],
        how="inner"
    )
    base_preenchimento = base_reduzida.groupBy(
        "Subcategoria", "Item de configuração", "Categoria"
    ).count()
    window_spec = Window.partitionBy(
        "Subcategoria", "Item de configuração"
    ).orderBy(F.desc("count"), F.asc("Categoria"))
    base_preenchimento = base_preenchimento.withColumn(
    "rank", F.row_number().over(window_spec)
    ).filter(F.col("rank") == 1).select(
        "Subcategoria",
        "Item de configuração",
        F.col("Categoria").alias("Categoria_sugerida")
    )
    df = df.join(
        base_preenchimento,
        on=["Subcategoria", "Item de configuração"],
        how="left"
    )
    df = df.withColumn(
        "Categoria",
        F.when(
            (F.col("Categoria").isNull()) &
            (F.col("Categoria_sugerida").isNotNull()),
            F.col("Categoria_sugerida")
        ).otherwise(F.col("Categoria"))
    )
    df = df.withColumn(
        "Modified_Record",
        F.when(
            (F.col("Subcategoria_without_Categoria") == True),
            True
        ).otherwise(F.col("Modified_Record"))
    )
    df.filter(filtro_registros).filter(F.col("Subcategoria_without_Categoria") == True).select(colunas_originais).show(20, truncate=False)
else:
    print("✅ Todos os registros com 'Subcategoria' preenchida também têm 'Categoria' preenchida.")

# Dropando colunas auxiliares
df = df.drop("Subcategoria_is_filled", "Categoria_is_filled", "Subcategoria_without_Categoria", "Categoria_sugerida")

⚠️ Encontrados 2 registros onde 'Subcategoria' está preenchida mas 'Categoria' está nula. Recomendado revisar esses casos.
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+----------------------------+--------------------------------------------------------+-------+-------------+-------------+-------------------------+----------------+------------+--------------+---------------+----------------------+-------------------+------------------------------+
|Número    |Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|Aberto             |Resolvido          |Encerrado          |Duração|Código de fechamento        |Descrição resumida                                      |Solução|Aberto por   |Incidente Pai|Status                   |Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|Subcategoria_is_filled|Categoria_is_filled|Subcategoria_without_Ca

#### Duração

In [89]:
# Validando os segundos de duração de um incidente

# Criando coluna de cálculo de segundos
# Quando "Resolvido" é preenchido, calcula a diferença entre "Aberto" e "Resolvido"
# Quando "Encerrado" é preenchido, calcula a diferença entre "Aberto"
df = df.withColumn(
    "Duração_Calculada",
    F.when(
        F.col("Resolvido").isNotNull(),
        F.unix_timestamp("Resolvido") - F.unix_timestamp("Aberto")
    ).when(
        F.col("Encerrado").isNotNull(),
        F.unix_timestamp("Encerrado") - F.unix_timestamp("Aberto")
    ).otherwise(None)
)
# Comparando "Duração_Calculada" com a coluna "Duração"
df = df.withColumn(
    "Duração_is_inconsistent",
    F.when(
        F.col("Duração").isNotNull() & F.col("Duração_Calculada").isNotNull(),
        F.abs(F.col("Duração") - F.col("Duração_Calculada")) > 60  # Considera inconsistente se a diferença for maior que 60 segundos
    ).otherwise(False)
)
total_inconsistentes = df.filter(filtro_registros).filter(F.col("Duração_is_inconsistent") == True).count()
if total_inconsistentes > 0:
    colunas_interesse = ["Número", "Status", "Aberto", "Resolvido", "Encerrado", "Duração", "Duração_Calculada", "Modified_Record"]
    print(f"⚠️ Encontrados {total_inconsistentes} registros ({total_inconsistentes/total_registros*100:.2f}%) onde 'Duração' é inconsistente com as datas de 'Aberto', 'Resolvido' e 'Encerrado'. Recomendado revisar esses casos.")
    df.filter(filtro_registros).filter(F.col("Duração_is_inconsistent") == True).select(colunas_interesse).show(5, truncate=False)
    # Determinando registros com inconsistência crítica (diferença maior que 5 minutos)
    df = df.withColumn(
        "Duração_is_critically_inconsistent",
        F.when(
            F.col("Duração_is_inconsistent") == True,
            F.abs(F.col("Duração") - F.col("Duração_Calculada")) > 300  # Considera criticamente inconsistente se a diferença for maior que 300 segundos (5 minutos)
        ).otherwise(False)
    )
    total_criticamente_inconsistentes = df.filter(filtro_registros).filter(F.col("Duração_is_critically_inconsistent") == True).count()
    print(f"⚠️ Desses, {total_criticamente_inconsistentes} registros ({total_criticamente_inconsistentes/total_registros*100:.2f}%) apresentam uma inconsistência crítica (diferença maior que 5 minutos).")
    df.filter(filtro_registros).filter(F.col("Duração_is_critically_inconsistent") == True).select(colunas_interesse).show(5, truncate=False)


    # Confirmando se todos os registros com "Duração" inconsistente tem "Resolvido" preenchido
    filtro_validacao = (F.col("Duração_Calculada") > F.col("Duração")) & (F.col("Resolvido").isNull()) & (F.col("Duração_is_inconsistent") == True)
    validacao_padrao_inconsistencia = df.filter(filtro_validacao)

    if validacao_padrao_inconsistencia.count() == total_inconsistentes:
        print("✅ Todos os registros com 'Duração' inconsistente tem 'Resolvido' nulo, o que é um padrão esperado para casos onde a duração informada não bate com as datas.\n Isso sugere que o nulo em 'Resolvido' pode ser um erro de preenchimento.")
        # Preenchendo a data de "Resolvido" com base nos registros que a duração calculada é menor que a duração informada
        # Nesses casos, é possível que o nulo de "Resolvido" seja um erro, e por isso não bateu com a duração informada. 
        # Assim, vamos preencher "Resolvido" com a data de "Aberto" + "Duração_Calculada", e marcar o registro como modificado.  
        df = df.withColumn(
            "Modified_Record",
            F.when(
                filtro_validacao,
                True
            ).otherwise(F.col("Modified_Record"))
        )
        df = df.withColumn(
            "Resolvido",
            F.when(
                filtro_validacao & (F.col("Status") != "Sem Intervenção"),
                F.from_unixtime(F.unix_timestamp("Aberto") + F.col("Duração"))
            ).otherwise(F.col("Resolvido"))
        )
        df = df.withColumn(
            "Duração",
            F.when(
                filtro_validacao & (F.col("Status") == "Sem Intervenção"),
                F.col("Duração_Calculada")
            ).otherwise(F.col("Duração"))
        )
        print("Inconsistências corrigidas preenchendo 'Resolvido' e ajustando 'Duração' para os casos de 'Sem Intervenção'.")
        registros_modificados = df.filter(filtro_registros).filter(F.col("Modified_Record") == True).filter(F.col("Duração_is_inconsistent") == True)
        print(f"Exibindo registros modificados nessa etapa (Total: {registros_modificados.count()}):")     
        df\
            .filter(filtro_registros)\
            .filter("Modified_Record = True")\
            .filter(F.col("Status") == "Sem Intervenção")\
            .select(colunas_interesse)\
            .show(5, truncate=False)
        df\
            .filter(filtro_registros)\
            .filter("Modified_Record = True")\
            .filter(F.col("Status") != "Sem Intervenção")\
            .select(colunas_interesse)\
            .show(5, truncate=False)
else:
    print("✅ Todos os registros tem 'Duração' consistente com as datas de 'Aberto', 'Resolvido' e 'Encerrado'.")

# Dropando colunas auxiliares
df = df.drop("Duração_Calculada", "Duração_is_inconsistent", "Duração_is_critically_inconsistent")

⚠️ Encontrados 271 registros (0.22%) onde 'Duração' é inconsistente com as datas de 'Aberto', 'Resolvido' e 'Encerrado'. Recomendado revisar esses casos.
+----------+---------------+-------------------+---------+-------------------+-------+-----------------+---------------+
|Número    |Status         |Aberto             |Resolvido|Encerrado          |Duração|Duração_Calculada|Modified_Record|
+----------+---------------+-------------------+---------+-------------------+-------+-----------------+---------------+
|INC8643285|Sem Intervenção|2025-12-22 10:39:01|NULL     |2025-12-22 12:17:57|5860   |5936             |false          |
|INC8643284|Sem Intervenção|2025-12-22 10:38:59|NULL     |2025-12-22 12:17:58|5862   |5939             |false          |
|INC8643283|Sem Intervenção|2025-12-22 10:38:57|NULL     |2025-12-22 12:17:58|5865   |5941             |false          |
|INC8643282|Sem Intervenção|2025-12-22 10:38:54|NULL     |2025-12-22 12:17:59|5869   |5945             |false          |

#### Entrou para KPI?

In [90]:
# Validando se "Entrou para KPI?" atende as regras

# Regra 1: Somente as prioridades 1, 2 e 3 entram para o KPI
# Regra 2: Incidente Pai com valor preenchido não entram no KPI
# Regra 3: Status = “Sem Intervenção”, não entram no KPI

df = df\
    .withColumn(
        "Entra_para_KPI",
        F.when(
            (F.col("Prioridade").isin("1 - Crítica", "2 - Alta", "3 - Média")) &
            (F.col("Incidente Pai").isNull()) &
            (F.col("Status") != "Sem Intervenção"),
            "SIM"
        ).otherwise("NAO")
    ).withColumn(
        "Entra_para_KPI_is_inconsistent",
        F.when(
            (F.col("Entra_para_KPI") != F.col("Entrou para KPI?")),
            True
        ).otherwise(False)
    )

total_inconsistentes_kpi = df.filter(filtro_registros).filter(F.col("Entra_para_KPI_is_inconsistent") == True).count()
if total_inconsistentes_kpi > 0:
    colunas_interesse = ["Número", "Prioridade", "Incidente Pai", "Status", "Entra_para_KPI", "Entrou para KPI?", "Modified_Record"]
    print(f"⚠️ Encontrados {total_inconsistentes_kpi} ({total_inconsistentes_kpi / total_registros * 100:.2f}%) registros onde 'Entrou para KPI?' é inconsistente com as regras definidas. Recomendado revisar esses casos.")
    df.filter(filtro_registros).filter(F.col("Entra_para_KPI_is_inconsistent") == True).select(colunas_interesse).show(20, truncate=False)
else:
    print("✅ Todos os registros tem 'Entrou para KPI?' consistente com as regras definidas.")
    
# Dropando colunas auxiliares
df = df.drop("Entra_para_KPI", "Entra_para_KPI_is_inconsistent")

⚠️ Encontrados 151 (0.12%) registros onde 'Entrou para KPI?' é inconsistente com as regras definidas. Recomendado revisar esses casos.
+----------+----------+-------------+-------------------------+--------------+----------------+---------------+
|Número    |Prioridade|Incidente Pai|Status                   |Entra_para_KPI|Entrou para KPI?|Modified_Record|
+----------+----------+-------------+-------------------------+--------------+----------------+---------------+
|INC8643410|3 - Média |NULL         |Encerrado Automaticamente|SIM           |NAO             |false          |
|INC8639258|3 - Média |NULL         |Encerrado Automaticamente|SIM           |NAO             |false          |
|INC8639255|3 - Média |NULL         |Encerrado Automaticamente|SIM           |NAO             |false          |
|INC8636352|3 - Média |NULL         |Encerrado                |SIM           |NAO             |false          |
|INC8634977|3 - Média |NULL         |Encerrado                |SIM           |NAO

#### KPI Violado?

In [91]:
# Validando se KPI Violado? atende as regras
# Regra 1 - Crítica - Duração até 4h
# Regra 2 - Alta - Duração até 4h
# Regra 3 - Média - Duração até 12h
# Regra 4 - Baixa - Duração até 24h
# Regra 5 - Muito Baixa - Duração até 96h

df = df.withColumn(
    "Duracao_Maxima_KPI",
    F.when(F.col("Prioridade") == "1 - Crítica", 4 * 3600)
     .when(F.col("Prioridade") == "2 - Alta", 4 * 3600)
     .when(F.col("Prioridade") == "3 - Média", 12 * 3600)
     .when(F.col("Prioridade") == "4 - Baixa", 24 * 3600)
     .when(F.col("Prioridade") == "5 - Muito Baixa", 96 * 3600)
)
df = df.withColumn(
    "KPI_Violado_Calculado",
    F.when(
        F.col("Entrou para KPI?") == "NAO",
        "N/A"
    ).when(
        F.col("Duração") > F.col("Duracao_Maxima_KPI"),
        "SIM"
    ).when(
        F.col("Duração") <= F.col("Duracao_Maxima_KPI"),
        "NAO"
    )
).withColumn(
    "KPI_Violado_is_inconsistent",
    F.when(
        (F.col("KPI_Violado_Calculado") != F.col("KPI Violado?")),
        True
    ).otherwise(False)
)

total_inconsistentes_kpi_violado = df.filter(filtro_registros).filter(F.col("KPI_Violado_is_inconsistent") == True).count()
if total_inconsistentes_kpi_violado > 0:
    colunas_interesse = ["Número", "Prioridade", "Duração", "Duracao_Maxima_KPI", "Entrou para KPI?", "KPI_Violado_Calculado", "KPI Violado?", "Modified_Record"]
    print(f"⚠️ Encontrados {total_inconsistentes_kpi_violado} ({total_inconsistentes_kpi_violado / total_registros * 100:.2f}%) registros onde 'KPI Violado?' é inconsistente com as regras definidas. Recomendado revisar esses casos.")
    df.filter(filtro_registros).filter(F.col("KPI_Violado_is_inconsistent") == True).select(colunas_interesse).show(20, truncate=False)
else:
    print("✅ Todos os registros tem 'KPI Violado?' consistente com as regras definidas.")

# Dropando colunas auxiliares
df = df.drop("KPI_Violado_Calculado", "KPI_Violado_is_inconsistent")

⚠️ Encontrados 3399 (2.77%) registros onde 'KPI Violado?' é inconsistente com as regras definidas. Recomendado revisar esses casos.
+----------+----------+-------+------------------+----------------+---------------------+------------+---------------+
|Número    |Prioridade|Duração|Duracao_Maxima_KPI|Entrou para KPI?|KPI_Violado_Calculado|KPI Violado?|Modified_Record|
+----------+----------+-------+------------------+----------------+---------------------+------------+---------------+
|INC8650576|3 - Média |97941  |43200             |SIM             |SIM                  |NAO         |false          |
|INC8650448|3 - Média |102691 |43200             |SIM             |SIM                  |NAO         |false          |
|INC8647625|2 - Alta  |37211  |14400             |SIM             |SIM                  |NAO         |false          |
|INC8647247|3 - Média |103049 |43200             |SIM             |SIM                  |NAO         |false          |
|INC8646341|3 - Média |336121 |4320

#### Status

In [92]:
# Validando o status de cada incidente
# Descobrindo a regra de negócio associada ao campo "Status"
df = df.withColumn(
    "Tempo_Pos_Resolução",
    F.when(
        (F.col("Encerrado").isNotNull()) & (F.col("Resolvido").isNotNull()) & (F.col("Encerrado") > F.col("Resolvido")),
        F.unix_timestamp("Encerrado") - F.unix_timestamp("Resolvido")
    ).otherwise(None)
)

df_base = df.filter(filtro_registros).filter(F.col("Status").isNotNull())
# Total geral (pra calcular proporção)
total_geral = df_base.count()

df_status = df_base.groupby("Status").agg(
    F.count("*").alias("Total"),
    # Percentual do total geral
    F.round((F.count("*") / total_geral)*100, 2).alias("% do Total"),
    # Percentuais internos (por status)
    F.round((F.sum(F.when(F.col("Resolvido").isNull(), 1).otherwise(0)) / F.count("*"))*100, 2).alias("% Resolvido Nulo"),
    F.round((F.sum(F.when(F.col("Aberto por") == "Monitoramento", 1).otherwise(0)) / F.count("*"))*100, 2).alias("% Monitoramento"),
    F.round((F.sum(F.when(F.col("Aberto por") == "Manual", 1).otherwise(0)) / F.count("*"))*100, 2).alias("% Manual"),
    # Estatísticas de Duração
    F.round(F.avg("Duração")).alias("Duração Média"),
    F.percentile_approx("Duração", 0.5).alias("Duração Mediana"),
    # Estatísticas de tempo pós resolução para os casos de "Encerrado"
    F.round(F.avg("Tempo_Pos_Resolução")).alias("Tempo Pós Resolução Médio"),
    F.percentile_approx("Tempo_Pos_Resolução", 0.5).alias("Tempo Pós Resolução Mediano")
)

# Ordenando
df_status.orderBy(F.desc("Total")).show(truncate=False)


+-------------------------+-----+----------+----------------+---------------+--------+-------------+---------------+-------------------------+---------------------------+
|Status                   |Total|% do Total|% Resolvido Nulo|% Monitoramento|% Manual|Duração Média|Duração Mediana|Tempo Pós Resolução Médio|Tempo Pós Resolução Mediano|
+-------------------------+-----+----------+----------------+---------------+--------+-------------+---------------+-------------------------+---------------------------+
|Sem Intervenção          |80364|65.59     |100.0           |99.96          |0.04    |19258.0      |334            |NULL                     |NULL                       |
|Encerrado Automaticamente|26830|21.9      |5.57            |59.1           |40.9    |997817.0     |7872           |392382.0                 |388562                     |
|Encerrado                |15337|12.52     |2.54            |52.81          |47.19   |139491.0     |3144           |28775.0                  |924

In [93]:
# Tempo pós resolução para os casos de "Encerrado" e "Encerrado Automaticamente"
df_base = df.filter(filtro_registros).filter(F.col("Status").isin("Encerrado", "Encerrado Automaticamente")).filter(F.col("Tempo_Pos_Resolução").isNotNull())
df_status = df_base.groupby("Status").agg(
        F.round(F.avg("Tempo_Pos_Resolução")).alias("Tempo Pós Resolução Médio"),
        F.min("Tempo_Pos_Resolução").alias("Tempo Pós Resolução Mínimo"),
        F.percentile_approx("Tempo_Pos_Resolução", 0.25).alias("Tempo Pós Resolução Q1"),
        F.percentile_approx("Tempo_Pos_Resolução", 0.5).alias("Tempo Pós Resolução Mediano"),
        F.percentile_approx("Tempo_Pos_Resolução", 0.75).alias("Tempo Pós Resolução Q3"),
        F.max("Tempo_Pos_Resolução").alias("Tempo Pós Resolução Máximo")
)
df_status.show(truncate=False)

+-------------------------+-------------------------+--------------------------+----------------------+---------------------------+----------------------+--------------------------+
|Status                   |Tempo Pós Resolução Médio|Tempo Pós Resolução Mínimo|Tempo Pós Resolução Q1|Tempo Pós Resolução Mediano|Tempo Pós Resolução Q3|Tempo Pós Resolução Máximo|
+-------------------------+-------------------------+--------------------------+----------------------+---------------------------+----------------------+--------------------------+
|Encerrado                |28775.0                  |7                         |276                   |924                        |12785                 |416528                    |
|Encerrado Automaticamente|392382.0                 |21                        |374660                |388562                     |403329                |40187518                  |
+-------------------------+-------------------------+--------------------------+----------

In [94]:
# Validando a regra de negócio inferida para o campo "Status" com base nos padrões observados:
# Usando critério de tempo pós resolução para diferenciar "Encerrado" de "Encerrado Automaticamente"
threshold_automatico = 24 * 3600  # 24h
df = df.withColumn(
    "Status_inferido",
    F.when(
        (F.col("Aberto por") == "Monitoramento") &
        (F.col("Resolvido").isNull()) &
        (F.col("Duração") < threshold_automatico),  # evita casos longos
        "Sem Intervenção"
    )
    .when(
        (F.col("Resolvido").isNotNull()) &
        (F.col("Tempo_Pos_Resolução").isNotNull()) &
        (F.col("Tempo_Pos_Resolução") < threshold_automatico),
        "Encerrado"
    )
    .when(
        (F.col("Tempo_Pos_Resolução").isNotNull()) &
        (F.col("Tempo_Pos_Resolução") >= threshold_automatico),
        "Encerrado Automaticamente"
    )
    .otherwise("Encerrado Automaticamente")
)
df_filtrado = df.filter(filtro_registros).filter(F.col("Status") != F.col("Status_inferido"))
validacao_regra_status = df_filtrado.count()
pct_fora_regra = round(validacao_regra_status / total_geral * 100, 2)
print(f"⚠️ Atenção: {validacao_regra_status} registros ({pct_fora_regra}%) estão fora da regra inferida para 'Status'.")
df_filtrado.select("Número", "Aberto por", "Resolvido", "Duração", "Status", "Status_inferido").show(20, truncate=False)

⚠️ Atenção: 2619 registros (2.14%) estão fora da regra inferida para 'Status'.
+----------+-------------+-------------------+-------+---------------+-------------------------+
|Número    |Aberto por   |Resolvido          |Duração|Status         |Status_inferido          |
+----------+-------------+-------------------+-------+---------------+-------------------------+
|INC8652635|Monitoramento|NULL               |95035  |Sem Intervenção|Encerrado Automaticamente|
|INC8652586|Monitoramento|NULL               |102554 |Sem Intervenção|Encerrado Automaticamente|
|INC8651368|Manual       |2025-12-29 10:44:26|8562   |Encerrado      |Encerrado Automaticamente|
|INC8651363|Manual       |2025-12-29 08:15:49|190    |Encerrado      |Encerrado Automaticamente|
|INC8651000|Manual       |2025-12-29 06:44:34|16328  |Encerrado      |Encerrado Automaticamente|
|INC8650448|Monitoramento|NULL               |102691 |Encerrado      |Encerrado Automaticamente|
|INC8650072|Monitoramento|NULL               |16

In [95]:
# Flag para registros com Resolvido > Encerrado
df = df.withColumn(
    "Resolvido_after_Encerrado",
    F.when(
        (F.col("Resolvido").isNotNull()) &
        (F.col("Encerrado").isNotNull()) &
        (F.col("Resolvido") > F.col("Encerrado")),
        True
    ).otherwise(False)
)
total_resolvido_after_encerrado = df.filter(filtro_registros).filter(F.col("Resolvido_after_Encerrado") == True).count()
if total_resolvido_after_encerrado > 0:
    print(f"⚠️ Encontrados {total_resolvido_after_encerrado} registros onde 'Resolvido' é posterior a 'Encerrado', o que é uma inconsistência temporal. Recomendado revisar esses casos.")
    df.filter(filtro_registros).filter(F.col("Resolvido_after_Encerrado") == True).select("Número", "Aberto", "Resolvido", "Encerrado", "Status").show(20, truncate=False)
else:
    print("✅ Nenhum registro encontrado com 'Resolvido' posterior a 'Encerrado'.")

⚠️ Encontrados 32 registros onde 'Resolvido' é posterior a 'Encerrado', o que é uma inconsistência temporal. Recomendado revisar esses casos.
+----------+-------------------+-------------------+-------------------+-------------------------+
|Número    |Aberto             |Resolvido          |Encerrado          |Status                   |
+----------+-------------------+-------------------+-------------------+-------------------------+
|INC8564686|2025-10-11 10:24:11|2025-10-14 10:33:32|2025-10-11 10:51:08|Encerrado Automaticamente|
|INC8564684|2025-10-11 10:22:09|2025-10-14 10:33:32|2025-10-11 10:51:07|Encerrado Automaticamente|
|INC8559567|2025-10-07 18:52:38|2025-10-09 12:26:33|2025-10-08 00:30:40|Encerrado Automaticamente|
|INC8559330|2025-10-07 15:13:22|2025-10-07 17:34:42|2025-10-07 15:50:23|Encerrado                |
|INC8557728|2025-10-06 14:09:20|2025-10-06 18:23:53|2025-10-06 15:26:58|Encerrado                |
|INC8524455|2025-09-03 17:41:59|2025-09-04 18:50:22|2025-09-03 19:

### Nulos

In [96]:
# Validar valores nulos em colunas que não deveriam ter - desconsiderando registros previamente marcados para remoção
colunas_nao_nulas = [
    "Número",
    "Prioridade",
    "Grupo designado",
    "Aberto",
    "Encerrado",
    "Duração",
    "Descrição Resumida",
    "Aberto por",
    "Status",
    "Entrou para KPI?",
    "KPI Violado?"
]

for coluna in colunas_nao_nulas:
    df = identificar_nulos(df, coluna)

for coluna in colunas_nao_nulas:
    nulos_count = df.filter(filtro_registros).filter(F.col(f"{coluna}_is_null") == True).count()
    if nulos_count > 0:
        print(f"⚠️ A coluna '{coluna}' contém {nulos_count} valores nulos - ({nulos_count/total_registros*100:.2f}%).")
        df.filter(filtro_registros).filter(F.col(f"{coluna}_is_null") == True).select(colunas_originais).show(nulos_count, truncate=False)
    else:
        print(f"✅ A coluna '{coluna}' não contém valores nulos.")
        # Drop da coluna auxiliar
        df = df.drop(f"{coluna}_is_null")

✅ A coluna 'Número' não contém valores nulos.
✅ A coluna 'Prioridade' não contém valores nulos.
✅ A coluna 'Grupo designado' não contém valores nulos.
✅ A coluna 'Aberto' não contém valores nulos.
✅ A coluna 'Encerrado' não contém valores nulos.
✅ A coluna 'Duração' não contém valores nulos.
✅ A coluna 'Descrição Resumida' não contém valores nulos.
⚠️ A coluna 'Aberto por' contém 11 valores nulos - (0.01%).
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+-----------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+-------+----------+-------------+------+----------------+------------+--------------+---------------+
|Número    |Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|Aberto             |Resolvido          |Encerrado          |Dur

#### Aberto por

In [97]:
# Aberto por
'''
Manual ou Monitoramento - Na Descrição Resumida, todos os registros que começam com "Problem: Alarm Application Monitoring" 
foram abertos por "Monitoramento". Portanto, é provável que os registros nulos tenham sido abertos da mesma forma, ou seja, por "Monitoramento".
'''
incidentes_similares = df\
    .filter(F.col("Descrição Resumida").startswith("Problem: Alarm Application Monitoring"))\
    .filter((F.col("Prioridade") != "4 - Baixa") & (F.col("Prioridade") != "5 - Muito Baixa"))\
    .filter(F.col("Aberto por").isNotNull())\
    .count()

incidentes_similares_monitoramento = df\
    .filter(F.col("Descrição Resumida").startswith("Problem: Alarm Application Monitoring"))\
    .filter((F.col("Prioridade") != "4 - Baixa") & (F.col("Prioridade") != "5 - Muito Baixa"))\
    .filter(F.col("Aberto por").isNotNull())\
    .filter(F.col("Aberto por")== "Monitoramento")\
    .count()

print(f"Total de incidentes com descrição e prioridade similar: {incidentes_similares}")
print(f"Total de incidentes com descrição e prioridade similar abertos por Monitoramento: {incidentes_similares_monitoramento}")
print(f"% de problemas similares abertos por Monitoramento: {(incidentes_similares_monitoramento/incidentes_similares)*100:.2f}%")

Total de incidentes com descrição e prioridade similar: 714
Total de incidentes com descrição e prioridade similar abertos por Monitoramento: 714
% de problemas similares abertos por Monitoramento: 100.00%


In [98]:
# Substituindo valores nulos em "Aberto por" por "Monitoramento", baseado na análise anterior
df = df.withColumn(
    "Modified_Record",
    F.when(F.col("Aberto por_is_null")==True, True).otherwise(F.col("Modified_Record"))
)
df = df.withColumn(
    "Aberto por",
    F.when((F.col("Aberto por_is_null")==True), "Monitoramento").otherwise(F.col("Aberto por"))
)
df.filter(filtro_registros).filter(F.col("Aberto por_is_null")==True).select(colunas_originais).show()

# Drop coluna auxiliar de nulos
df = df.drop("Aberto por_is_null")

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+------+----------------+------------+--------------+---------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|          Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|Status|Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+------+----------------+------------+--------------+---------------+
|INC8642461| 3 - Média|   NULL|     NULL|        NULL|         Team14|            

#### Status

In [99]:
# Status
'''
Usando a regra inferida
'''
# Preencher a regra inferida novamente (considerando o aberto por monitoramento preenchido)
df = df.withColumn(
    "Status_inferido",
    F.when(
        (F.col("Status_is_null") == True) &
        (F.col("Aberto por") == "Monitoramento") &
        (F.col("Resolvido").isNull()) &
        (F.col("Duração") < threshold_automatico),  # evita casos longos
        "Sem Intervenção"
    )
    .when(
        (F.col("Status_is_null") == True) &
        (F.col("Resolvido").isNotNull()) &
        (F.col("Tempo_Pos_Resolução").isNotNull()) &
        (F.col("Tempo_Pos_Resolução") < threshold_automatico),
        "Encerrado"
    )
    .when(
        (F.col("Status_is_null") == True) &
        (F.col("Tempo_Pos_Resolução").isNotNull()) &
        (F.col("Tempo_Pos_Resolução") >= threshold_automatico),
        "Encerrado Automaticamente"
    )
    .otherwise("Status_inferido")
)
# Resolvido Nulo ou Não X Status "Sem Intervenção"
df = df.withColumn(
    "Modified_Record",
    F.when(F.col("Status_is_null") == True, True).otherwise(F.col("Modified_Record"))
)
df = df.withColumn(
    "Status",
    F.when(F.col("Status_is_null") == True, F.col("Status_inferido")).otherwise(F.col("Status"))
)
df.filter(filtro_registros).filter(F.col("Status_is_null") == True).select(colunas_originais).show()

# Drop coluna auxiliar de nulos e inferido
df = df.drop("Status_is_null")

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|          Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|INC8642461| 3 - Média|   NULL|     NULL|        NULL| 

#### Entrou para KPI?

In [100]:
# Entrou para KPI?
'''
Somente as prioridades 1, 2 e 3 entram para o KPI
Campo: Incidente Pai com valor preenchido não entram no KPI
Campo: Status = “Sem Intervenção”, não entram no KPI
'''
df = df.withColumn(
    "Modified_Record",
    F.when(F.col("Entrou para KPI?_is_null") == True, True).otherwise(F.col("Modified_Record"))
)
df = df.withColumn(
    "Entrou para KPI?",
    F.when(
        (F.col("Entrou para KPI?_is_null") == True) &
        (F.col("Prioridade").isin("1 - Crítica", "2 - Alta", "3 - Média")) &
        (F.col("Incidente Pai").isNull()) &
        (F.col("Status") != "Sem Intervenção"),
        "SIM"
    ).when(
        (F.col("Entrou para KPI?_is_null") == True) &
        (
            (F.col("Prioridade").isin("4 - Baixa", "5 - Muito Baixa")) |
            (F.col("Incidente Pai").isNotNull()) |
            (F.col("Status") == "Sem Intervenção")
        ),
        "NAO"
    ).otherwise(F.col("Entrou para KPI?"))
)

df.filter(filtro_registros).filter(F.col("Entrou para KPI?_is_null") == True).select(colunas_originais).show()

# Drop coluna auxiliar de nulos
df = df.drop("Entrou para KPI?_is_null")

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|          Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|INC8642461| 3 - Média|   NULL|     NULL|        NULL| 

#### KPI Violado?

In [101]:
# KPI Violado?
'''
1 - Crítica - Duração até 4h
2 - Alta - Duração até 4h
3 - Média - Duração até 12h
4 - Baixa - Duração até 24h
5 - Muito Baixa - Duração até 96h
'''
df = df.withColumn(
    "Modified_Record",
    F.when(F.col("KPI Violado?_is_null") == True, True).otherwise(F.col("Modified_Record"))
)
df = df.withColumn(
    "KPI Violado?",
    F.when(
        (F.col("KPI Violado?_is_null") == True) &
        (F.col("Entrou para KPI?") == "NAO"),
        "N/A"
    ).when(
        (F.col("KPI Violado?_is_null") == True) &
        (F.col("Duração") > F.col("Duracao_Maxima_KPI")),
        "SIM"
    ).when(
        (F.col("KPI Violado?_is_null") == True) &
        (F.col("Duração") <= F.col("Duracao_Maxima_KPI")),
        "NAO"
    ).otherwise(F.col("KPI Violado?"))
)

df.filter(filtro_registros).filter(F.col("KPI Violado?_is_null") == True).select(colunas_originais).show()

# Drop coluna auxiliar de nulos
df = df.drop("KPI Violado?_is_null")

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|          Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|INC8642461| 3 - Média|   NULL|     NULL|        NULL| 

### Relatório Final - Qualidade da Base

In [102]:
# Relatório final

# Número de registros marcados para remoção
drop_data = df.filter(F.col("Drop_from_base") == True)
total_para_remocao = drop_data.count()
print(f"Total de registros marcados para remoção: {total_para_remocao} ({total_para_remocao/(total_registros+total_para_remocao)*100:.2f}%)")
drop_data.select(colunas_originais).show(20, truncate=False)

# Número de registros modificados
modified_data = df.filter(filtro_registros).filter(F.col("Modified_Record") == True)
total_modificados = modified_data.count()
print(f"Total de registros modificados: {total_modificados} ({total_modificados/total_registros*100:.2f}%)")
modified_data.select(colunas_originais).show(20, truncate=False)

Total de registros marcados para remoção: 11 (0.01%)
+-------------------------------------------------------------------------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------+-------+--------------------+------------------+-------+-------------+-------------+---------------+----------------+------------+--------------+---------------+
|Número                                                                   |Prioridade|Produto      |Categoria |Subcategoria   |Grupo designado|Item de configuração|Aberto|Resolvido|Encerrado|Duração|Código de fechamento|Descrição resumida|Solução|Aberto por   |Incidente Pai|Status         |Entrou para KPI?|KPI Violado?|Drop_from_base|Modified_Record|
+-------------------------------------------------------------------------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------+-------+--------------------+------------------+--

## Salvando dados tratados na camada bronze

In [103]:
df.columns

['Subcategoria',
 'Item de configuração',
 'Número',
 'Prioridade',
 'Produto',
 'Categoria',
 'Grupo designado',
 'Aberto',
 'Resolvido',
 'Encerrado',
 'Duração',
 'Código de fechamento',
 'Descrição resumida',
 'Solução',
 'Aberto por',
 'Incidente Pai',
 'Status',
 'Entrou para KPI?',
 'KPI Violado?',
 'Drop_from_base',
 'Modified_Record',
 'Duracao_Maxima_KPI',
 'Tempo_Pos_Resolução',
 'Status_inferido',
 'Resolvido_after_Encerrado']

In [104]:
# Save do arquivo tratado
df_final = df\
    .filter(filtro_registros)\
    .select(
        "Número",
        "Prioridade",
        "Produto",
        "Categoria",
        "Subcategoria",
        "Grupo designado",
        "Item de configuração",
        "Aberto",
        "Resolvido",
        "Encerrado",
        "Duração",
        "Descrição Resumida",
        "Solução",
        "Aberto por",
        "Incidente Pai",
        "Status",
        "Entrou para KPI?",
        "KPI Violado?",
        "Tempo_Pos_Resolução",
        "Resolvido_after_Encerrado",
        "Modified_Record"
    )

# Salvando o DataFrame final como CSV
df_final.write\
    .mode("overwrite")\
    .option("header", True)\
    .option("delimiter", ";")\
    .options(encoding="ISO-8859-1")\
    .csv(f"{SAVE_PATH}/b_incidentes.csv")